# LSTM Predictive Maintenance - Training & Evaluation

Notebook ini dibagi menjadi:
1. **Setup** - Install dependencies & import
2. **Data Loading** - Load dan preprocess data
3. **Model Definition** - LSTM dengan CuPy (GPU)
4. **Training** - Training dengan validation tracking
5. **Testing/Prediction** - Evaluasi model

---
## 1. Setup

In [ ]:
# Uncomment untuk install di Colab
# !pip install cupy-cuda12x

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import os

# Check GPU
try:
    import cupy as cp
    GPU_AVAILABLE = True
    print("✅ CuPy detected - Using GPU")
except ImportError:
    import numpy as cp
    GPU_AVAILABLE = False
    print("⚠️ CuPy not found - Using CPU")

In [ ]:
# Hyperparameters
SEQUENCE_LENGTH = 5
HIDDEN_SIZE = 128
EPOCHS = 250
LEARNING_RATE = 0.001
BATCH_SIZE = 128
TEST_SPLIT = 0.2
VAL_SPLIT = 0.1
EARLY_STOPPING_PATIENCE = 15

# Paths
DATA_DIR = "../data"
INPUT_FILE = os.path.join(DATA_DIR, "labeled_dataset.csv")

---
## 2. Data Loading & Preprocessing

In [ ]:
def preprocess_data(filepath, sequence_length=5):
    """Load and preprocess data into sequences."""
    print("Loading data...")
    df = pd.read_csv(filepath)
    print(f"Loaded {len(df)} rows")
    
    # Select features
    feature_cols = ['Service Type', 'Service Name', 'Type', 'Status', 'SLA (minutes)', 'Month']
    df_features = df[feature_cols].copy()
    y = df['label'].values
    
    # Encode categorical
    for col in ['Service Type', 'Service Name', 'Type', 'Status', 'Month']:
        df_features[col] = df_features[col].fillna('Unknown')
        le = LabelEncoder()
        df_features[col] = le.fit_transform(df_features[col].astype(str))
    
    df_features['SLA (minutes)'] = df_features['SLA (minutes)'].fillna(df_features['SLA (minutes)'].median())
    
    # Scale
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(df_features.values)
    
    # Create sequences
    X_seq, y_seq = [], []
    for i in range(len(X_scaled) - sequence_length):
        X_seq.append(X_scaled[i:i + sequence_length])
        y_seq.append(y[i + sequence_length])
    
    print(f"Sequences: {len(X_seq)}, Shape: ({sequence_length}, {X_scaled.shape[1]})")
    return np.array(X_seq), np.array(y_seq)

In [ ]:
# Load data
X, y = preprocess_data(INPUT_FILE, SEQUENCE_LENGTH)

# Split: Train / Validation / Test
test_idx = int(len(X) * (1 - TEST_SPLIT))
X_trainval, X_test = X[:test_idx], X[test_idx:]
y_trainval, y_test = y[:test_idx], y[test_idx:]

val_idx = int(len(X_trainval) * (1 - VAL_SPLIT))
X_train, X_val = X_trainval[:val_idx], X_trainval[val_idx:]
y_train, y_val = y_trainval[:val_idx], y_trainval[val_idx:]

print(f"\nTrain: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

---
## 3. Model Definition

In [ ]:
class LSTMModel:
    """Batched LSTM with Adam optimizer."""
    
    def __init__(self, input_size, hidden_size, output_size=1):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        
        z_dim = hidden_size + input_size
        scale = 0.1
        
        self.params = {
            'Wf': cp.random.randn(hidden_size, z_dim).astype(cp.float32) * scale,
            'Wi': cp.random.randn(hidden_size, z_dim).astype(cp.float32) * scale,
            'Wc': cp.random.randn(hidden_size, z_dim).astype(cp.float32) * scale,
            'Wo': cp.random.randn(hidden_size, z_dim).astype(cp.float32) * scale,
            'Wy': cp.random.randn(output_size, hidden_size).astype(cp.float32) * scale,
            'bf': cp.zeros((hidden_size, 1), dtype=cp.float32),
            'bi': cp.zeros((hidden_size, 1), dtype=cp.float32),
            'bc': cp.zeros((hidden_size, 1), dtype=cp.float32),
            'bo': cp.zeros((hidden_size, 1), dtype=cp.float32),
            'by': cp.zeros((output_size, 1), dtype=cp.float32),
        }
        
        self.m = {k: cp.zeros_like(v) for k, v in self.params.items()}
        self.v = {k: cp.zeros_like(v) for k, v in self.params.items()}
        self.t = 0

    def sigmoid(self, x):
        return 1 / (1 + cp.exp(-cp.clip(x, -500, 500)))

    def forward_batch(self, X_batch):
        batch_size, seq_len, _ = X_batch.shape
        h = cp.zeros((self.hidden_size, batch_size), dtype=cp.float32)
        c = cp.zeros((self.hidden_size, batch_size), dtype=cp.float32)
        caches = []
        
        for t in range(seq_len):
            x_t = X_batch[:, t, :].T
            z = cp.vstack((h, x_t))
            
            f = self.sigmoid(cp.dot(self.params['Wf'], z) + self.params['bf'])
            i = self.sigmoid(cp.dot(self.params['Wi'], z) + self.params['bi'])
            c_bar = cp.tanh(cp.dot(self.params['Wc'], z) + self.params['bc'])
            o = self.sigmoid(cp.dot(self.params['Wo'], z) + self.params['bo'])
            
            c_new = f * c + i * c_bar
            h_new = o * cp.tanh(c_new)
            
            caches.append((z, f, i, c_bar, c_new, o, h_new, c, h))
            h, c = h_new, c_new
        
        y_pred = self.sigmoid(cp.dot(self.params['Wy'], h) + self.params['by'])
        return y_pred.flatten(), caches, h, c

    def backward_batch(self, y_pred, y_true, caches):
        batch_size = len(y_true)
        seq_len = len(caches)
        grads = {k: cp.zeros_like(v) for k, v in self.params.items()}
        
        dy = (y_pred - y_true).reshape(1, -1)
        h_final = caches[-1][6]
        
        grads['Wy'] = cp.dot(dy, h_final.T) / batch_size
        grads['by'] = cp.sum(dy, axis=1, keepdims=True) / batch_size
        
        dh_next = cp.dot(self.params['Wy'].T, dy)
        dc_next = cp.zeros((self.hidden_size, batch_size), dtype=cp.float32)
        
        for t in reversed(range(seq_len)):
            z, f, i, c_bar, c, o, h, c_prev, h_prev = caches[t]
            dh = dh_next
            
            do = dh * cp.tanh(c)
            da_o = do * o * (1 - o)
            grads['Wo'] += cp.dot(da_o, z.T) / batch_size
            grads['bo'] += cp.sum(da_o, axis=1, keepdims=True) / batch_size
            
            dc = dh * o * (1 - cp.tanh(c)**2) + dc_next
            
            dc_bar = dc * i
            da_c = dc_bar * (1 - c_bar**2)
            grads['Wc'] += cp.dot(da_c, z.T) / batch_size
            grads['bc'] += cp.sum(da_c, axis=1, keepdims=True) / batch_size
            
            di = dc * c_bar
            da_i = di * i * (1 - i)
            grads['Wi'] += cp.dot(da_i, z.T) / batch_size
            grads['bi'] += cp.sum(da_i, axis=1, keepdims=True) / batch_size
            
            df = dc * c_prev
            da_f = df * f * (1 - f)
            grads['Wf'] += cp.dot(da_f, z.T) / batch_size
            grads['bf'] += cp.sum(da_f, axis=1, keepdims=True) / batch_size
            
            dz = (cp.dot(self.params['Wf'].T, da_f) + cp.dot(self.params['Wi'].T, da_i) +
                  cp.dot(self.params['Wc'].T, da_c) + cp.dot(self.params['Wo'].T, da_o))
            
            dh_next = dz[:self.hidden_size, :]
            dc_next = f * dc
        
        for k in grads:
            grads[k] = cp.clip(grads[k], -5, 5)
        return grads

    def update_adam(self, grads, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        self.t += 1
        for k in self.params:
            self.m[k] = beta1 * self.m[k] + (1 - beta1) * grads[k]
            self.v[k] = beta2 * self.v[k] + (1 - beta2) * (grads[k] ** 2)
            m_hat = self.m[k] / (1 - beta1 ** self.t)
            v_hat = self.v[k] / (1 - beta2 ** self.t)
            self.params[k] -= lr * m_hat / (cp.sqrt(v_hat) + eps)

    def predict(self, X, batch_size=256):
        X_gpu = cp.asarray(X, dtype=cp.float32)
        preds = []
        for start in range(0, len(X_gpu), batch_size):
            end = min(start + batch_size, len(X_gpu))
            y_pred, _, _, _ = self.forward_batch(X_gpu[start:end])
            if GPU_AVAILABLE:
                preds.extend(cp.asnumpy(y_pred).tolist())
            else:
                preds.extend(y_pred.tolist())
        return np.array(preds)

In [ ]:
# Create model
input_size = X.shape[2]
model = LSTMModel(input_size=input_size, hidden_size=HIDDEN_SIZE)
print(f"Model created: LSTM(input={input_size}, hidden={HIDDEN_SIZE})")

---
## 4. Training (with Validation)

In [ ]:
def evaluate_loss(model, X, y, batch_size=128):
    """Calculate loss without training."""
    X_gpu = cp.asarray(X, dtype=cp.float32)
    y_gpu = cp.asarray(y, dtype=cp.float32)
    n_batches = (len(X_gpu) + batch_size - 1) // batch_size
    total_loss = 0
    
    for i in range(n_batches):
        start, end = i * batch_size, min((i + 1) * batch_size, len(X_gpu))
        y_pred, _, _, _ = model.forward_batch(X_gpu[start:end])
        loss = -cp.mean(y_gpu[start:end] * cp.log(y_pred + 1e-9) + 
                       (1 - y_gpu[start:end]) * cp.log(1 - y_pred + 1e-9))
        total_loss += float(loss)
    return total_loss / n_batches

In [ ]:
# Training loop
X_gpu = cp.asarray(X_train, dtype=cp.float32)
y_gpu = cp.asarray(y_train, dtype=cp.float32)
n_batches = (len(X_gpu) + BATCH_SIZE - 1) // BATCH_SIZE

history = {'train_loss': [], 'val_loss': []}
best_val_loss = float('inf')
patience_counter = 0

print(f"Training: {len(X_train)} samples, {n_batches} batches/epoch")
print("-" * 60)

for epoch in range(EPOCHS):
    # Shuffle
    idx = cp.random.permutation(len(X_gpu))
    X_shuf, y_shuf = X_gpu[idx], y_gpu[idx]
    
    epoch_loss = 0
    for i in range(n_batches):
        start, end = i * BATCH_SIZE, min((i + 1) * BATCH_SIZE, len(X_gpu))
        X_batch, y_batch = X_shuf[start:end], y_shuf[start:end]
        
        # Forward
        y_pred, caches, _, _ = model.forward_batch(X_batch)
        loss = -cp.mean(y_batch * cp.log(y_pred + 1e-9) + (1 - y_batch) * cp.log(1 - y_pred + 1e-9))
        epoch_loss += float(loss)
        
        # Backward & Update
        grads = model.backward_batch(y_pred, y_batch, caches)
        model.update_adam(grads, lr=LEARNING_RATE)
    
    # Calculate losses
    train_loss = epoch_loss / n_batches
    val_loss = evaluate_loss(model, X_val, y_val, BATCH_SIZE)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        status = "✓ Best"
    else:
        patience_counter += 1
        status = f"({patience_counter}/{EARLY_STOPPING_PATIENCE})"
    
    if (epoch + 1) % 10 == 0 or patience_counter == 0:
        print(f"Epoch {epoch+1:3d} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | {status}")
    
    if patience_counter >= EARLY_STOPPING_PATIENCE:
        print(f"\n🛑 Early stopping at epoch {epoch+1}")
        break
    
    if GPU_AVAILABLE:
        cp.cuda.Stream.null.synchronize()

print("-" * 60)
print(f"Training complete! Best val loss: {best_val_loss:.4f}")

In [ ]:
# Plot training history
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.grid(True)
plt.show()

# Check overfitting
gap = history['val_loss'][-1] - history['train_loss'][-1]
print(f"\nTrain-Val Gap: {gap:.4f}")
if gap > 0.1:
    print("⚠️ OVERFITTING detected")
elif gap > 0.05:
    print("⚡ Slight overfitting")
else:
    print("✅ Good fit")

---
## 5. Testing / Prediction

In [ ]:
# Generate predictions on test set
print("Generating predictions on test set...")
probabilities = model.predict(X_test, batch_size=BATCH_SIZE)
predictions = (probabilities >= 0.5).astype(int)

print(f"Test samples: {len(X_test)}")
print(f"Predictions: {len(predictions)}")

In [ ]:
# Calculate metrics
accuracy = np.mean(predictions == y_test)

tp = np.sum((predictions == 1) & (y_test == 1))
fp = np.sum((predictions == 1) & (y_test == 0))
fn = np.sum((predictions == 0) & (y_test == 1))
tn = np.sum((predictions == 0) & (y_test == 0))

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("=" * 50)
print("TEST RESULTS")
print("=" * 50)
print(f"Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print("=" * 50)

In [ ]:
# Confusion Matrix
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

cm = confusion_matrix(y_test, predictions)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Anomaly'],
            yticklabels=['Normal', 'Anomaly'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
# Classification Report
print(classification_report(y_test, predictions, target_names=['Normal', 'Anomaly']))

In [ ]:
# Save results
results_df = pd.DataFrame({
    'actual': y_test,
    'predicted': predictions,
    'probability': probabilities
})

results_df.to_csv(os.path.join(DATA_DIR, 'predictions.csv'), index=False)
pd.DataFrame(history).to_csv(os.path.join(DATA_DIR, 'training_history.csv'), index=False)

print("✅ Results saved!")
results_df.head(10)